# Advanced Linear Algebra for Machine Learning

**Soft prerequisite** -- helpful background that will be reinforced during the course (Session 19: PCA).

**What this covers:** Eigenvalues, eigenvectors, symmetric matrices, Singular Value Decomposition (SVD), and their connection to PCA.

**Who it's for:** Students who are comfortable with basic linear algebra (vectors, matrices, dot products, matrix multiplication) and want to build intuition for the decompositions used heavily in ML.

**Estimated time:** 30--45 minutes.

**Requirements:** `numpy`, `matplotlib`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

---
## 1. Eigenvalues and Eigenvectors

### Definition

Given a square matrix $A \in \mathbb{R}^{n \times n}$, a non-zero vector $\mathbf{v}$ is an **eigenvector** of $A$ if:

$$A\mathbf{v} = \lambda \mathbf{v}$$

where $\lambda$ is the corresponding **eigenvalue**.

### Geometric intuition

Most vectors change direction when multiplied by a matrix. Eigenvectors are special: they only get **scaled** (stretched or flipped), never rotated. The eigenvalue $\lambda$ tells you the scaling factor.

- $\lambda > 1$: the vector is stretched.
- $0 < \lambda < 1$: the vector is compressed.
- $\lambda < 0$: the vector is flipped and scaled.
- $\lambda = 0$: the vector is sent to zero (the matrix is singular along that direction).

### Finding eigenvalues analytically

Eigenvalues satisfy the **characteristic equation**:

$$\det(A - \lambda I) = 0$$

For a $2 \times 2$ matrix $A = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$, this gives:

$$\lambda^2 - (a+d)\lambda + (ad - bc) = 0$$

### Example: Visualizing eigenvectors

In [ ]:
A = np.array([[2, 1],
              [1, 3]])

eigenvalues, eigenvectors = np.linalg.eig(A)
print("Eigenvalues:", eigenvalues)
print("Eigenvectors (columns):")
print(eigenvectors)

In [ ]:
# Verify: A @ v should equal lambda * v
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    lam = eigenvalues[i]
    print(f"Eigenvector {i}: {v}")
    print(f"  A @ v     = {A @ v}")
    print(f"  lambda * v = {lam * v}")
    print(f"  Match: {np.allclose(A @ v, lam * v)}\n")

In [ ]:
# Visualize: plot original vectors and their transformations
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: a random vector changes direction
ax = axes[0]
ax.set_title("Random vector: direction changes")
v_rand = np.array([1, 0.5])
Av_rand = A @ v_rand
ax.quiver(0, 0, v_rand[0], v_rand[1], angles='xy', scale_units='xy', scale=1, color='blue', label='v')
ax.quiver(0, 0, Av_rand[0], Av_rand[1], angles='xy', scale_units='xy', scale=1, color='red', label='Av')
ax.set_xlim(-1, 5)
ax.set_ylim(-1, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()

# Right: eigenvectors only get scaled
ax = axes[1]
ax.set_title("Eigenvectors: only scaled, not rotated")
colors = ['blue', 'green']
for i in range(2):
    v = eigenvectors[:, i]
    Av = A @ v
    ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
              color=colors[i], label=f'v{i} (lam={eigenvalues[i]:.2f})')
    ax.quiver(0, 0, Av[0], Av[1], angles='xy', scale_units='xy', scale=1,
              color=colors[i], alpha=0.4, linestyle='dashed', label=f'Av{i}')
ax.set_xlim(-2, 4)
ax.set_ylim(-2, 4)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

### Exercise 1.1

Given $B = \begin{pmatrix} 4 & 2 \\ 1 & 3 \end{pmatrix}$:

1. Compute the eigenvalues by hand using the characteristic equation $\lambda^2 - 7\lambda + 10 = 0$.
2. Verify your answer with NumPy.

In [ ]:
# YOUR CODE HERE


### Exercise 1.2

Create a $3 \times 3$ matrix of your choice. Compute its eigenvalues and eigenvectors with `np.linalg.eig`. Verify the relationship $A\mathbf{v} = \lambda \mathbf{v}$ for each eigenpair using `np.allclose`.

In [ ]:
# YOUR CODE HERE


---
## 2. Symmetric Matrices and Their Properties

A matrix $A$ is **symmetric** if $A = A^T$. Symmetric matrices are everywhere in ML: covariance matrices, kernel matrices, Hessians.

### Key properties

For a real symmetric matrix $A \in \mathbb{R}^{n \times n}$:

1. **All eigenvalues are real** (no complex numbers).
2. **Eigenvectors are orthogonal** (perpendicular to each other).
3. $A$ can be decomposed as $A = Q \Lambda Q^T$, where $Q$ is an orthogonal matrix (columns are eigenvectors) and $\Lambda$ is a diagonal matrix of eigenvalues. This is called the **spectral decomposition**.

### NumPy: `eig` vs `eigh`

- `np.linalg.eig` -- general eigendecomposition (eigenvalues may be complex).
- `np.linalg.eigh` -- specialized for symmetric (Hermitian) matrices. Faster, numerically more stable, and guarantees sorted real eigenvalues.

In [ ]:
# Create a symmetric matrix
S = np.array([[4, 2, 1],
              [2, 5, 3],
              [1, 3, 6]])

print("Symmetric?", np.allclose(S, S.T))

eigenvalues, Q = np.linalg.eigh(S)
print("\nEigenvalues (real, sorted):", eigenvalues)
print("\nEigenvectors (orthogonal columns):")
print(Q)

In [ ]:
# Verify orthogonality: Q^T Q should be the identity
print("Q^T @ Q (should be identity):")
print(Q.T @ Q)

# Verify spectral decomposition: S = Q Lambda Q^T
Lambda = np.diag(eigenvalues)
S_reconstructed = Q @ Lambda @ Q.T
print("\nReconstruction matches:", np.allclose(S, S_reconstructed))

### Exercise 2.1

Generate a random symmetric matrix: create a random $4 \times 4$ matrix `M` and compute `S = M + M.T` (this guarantees symmetry). Use `np.linalg.eigh` to verify that all eigenvalues are real and the eigenvectors are orthogonal.

In [ ]:
# YOUR CODE HERE


---
## 3. Singular Value Decomposition (SVD)

### What it is

The SVD factorizes **any** matrix $A \in \mathbb{R}^{m \times n}$ (not just square ones) as:

$$A = U \Sigma V^T$$

where:
- $U \in \mathbb{R}^{m \times m}$ -- orthogonal matrix (left singular vectors).
- $\Sigma \in \mathbb{R}^{m \times n}$ -- diagonal matrix of **singular values** $\sigma_1 \geq \sigma_2 \geq \ldots \geq 0$.
- $V \in \mathbb{R}^{n \times n}$ -- orthogonal matrix (right singular vectors).

### Intuition: generalized eigendecomposition

- The eigendecomposition $A = Q \Lambda Q^T$ only works for square (symmetric) matrices.
- SVD works for **any** matrix. It decomposes a linear transformation into: rotate ($V^T$) $\rightarrow$ scale ($\Sigma$) $\rightarrow$ rotate ($U$).
- Connection: the singular values of $A$ are the square roots of the eigenvalues of $A^T A$ (or $A A^T$).

### SVD with NumPy

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])

U, sigma, Vt = np.linalg.svd(A)

print("U (2x2):")
print(U)
print("\nSingular values:", sigma)
print("\nV^T (3x3):")
print(Vt)

In [ ]:
# Reconstruct A from SVD
# Note: np.linalg.svd returns sigma as a 1-D array, not a matrix.
# We need to build the full Sigma matrix.
Sigma = np.zeros_like(A, dtype=float)
np.fill_diagonal(Sigma, sigma)

A_reconstructed = U @ Sigma @ Vt
print("Original A:")
print(A)
print("\nReconstructed A:")
print(A_reconstructed)
print("\nMatch:", np.allclose(A, A_reconstructed))

In [ ]:
# Verify: singular values = sqrt(eigenvalues of A^T A)
ATA = A.T @ A
eig_ATA = np.linalg.eigvalsh(ATA)  # symmetric, so use eigvalsh
eig_ATA_sorted = np.sort(eig_ATA)[::-1]  # descending

print("Singular values of A:      ", sigma)
print("sqrt(eigenvalues of A^T A):", np.sqrt(eig_ATA_sorted[:len(sigma)]))

### Exercise 3.1

Create a $3 \times 4$ random matrix. Compute its SVD. Reconstruct the matrix from $U$, $\Sigma$, $V^T$ and verify the reconstruction is correct.

In [ ]:
# YOUR CODE HERE


### Exercise 3.2

For the same matrix, verify that the singular values equal the square roots of the eigenvalues of $A^T A$.

In [ ]:
# YOUR CODE HERE


---
## 4. Connection to PCA (Preview)

**Principal Component Analysis** finds the directions of maximum variance in data. Here is the key connection:

1. Given data matrix $X \in \mathbb{R}^{n \times d}$ (centered, i.e., mean subtracted), the **covariance matrix** is:

$$C = \frac{1}{n-1} X^T X$$

2. $C$ is symmetric and positive semi-definite, so its eigenvalues are $\geq 0$.

3. The **eigenvectors** of $C$ are the principal components (directions of max variance).

4. The **eigenvalues** of $C$ tell you how much variance each component explains.

5. Equivalently, if $X = U \Sigma V^T$ (SVD), then the columns of $V$ are the principal components, and the singular values relate to variance as $\text{variance}_i = \sigma_i^2 / (n-1)$.

We will build PCA from scratch in Session 19. For now, here is a quick demo.

In [ ]:
# Generate 2D data with clear principal directions
np.random.seed(42)
n = 200
# Data stretched along a diagonal direction
X = np.random.randn(n, 2) @ np.array([[3, 1], [1, 1]])

# Center the data
X_centered = X - X.mean(axis=0)

# Covariance matrix
C = (X_centered.T @ X_centered) / (n - 1)
print("Covariance matrix:")
print(C)

# Eigendecomposition of covariance
eig_vals, eig_vecs = np.linalg.eigh(C)
# Sort descending
idx = np.argsort(eig_vals)[::-1]
eig_vals = eig_vals[idx]
eig_vecs = eig_vecs[:, idx]

print("\nEigenvalues (variance per component):", eig_vals)
print("Variance explained ratio:", eig_vals / eig_vals.sum())

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.3, s=15)

# Plot principal components as arrows (scaled by sqrt of eigenvalue for visibility)
colors = ['red', 'blue']
for i in range(2):
    scale = np.sqrt(eig_vals[i]) * 2
    ax.quiver(0, 0, eig_vecs[0, i] * scale, eig_vecs[1, i] * scale,
              angles='xy', scale_units='xy', scale=1,
              color=colors[i], linewidth=2,
              label=f'PC{i+1} (var={eig_vals[i]:.2f})')

ax.set_aspect('equal')
ax.set_title('PCA: Eigenvectors of the Covariance Matrix')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## 5. Low-Rank Approximation with SVD (Image Compression)

One of the most practical uses of SVD: **low-rank approximation**.

Given $A = U \Sigma V^T$, the best rank-$k$ approximation (in the least-squares sense) is:

$$A_k = \sum_{i=1}^{k} \sigma_i \, \mathbf{u}_i \, \mathbf{v}_i^T$$

This is the **Eckart--Young theorem**. In practice, you keep only the top $k$ singular values and their corresponding vectors.

### Image compression demo

A grayscale image is just a matrix of pixel values. We can compress it by keeping only the top-$k$ singular values.

In [ ]:
# Create a sample grayscale image (synthetic)
np.random.seed(0)
# A structured image: gradient + some pattern
x = np.linspace(0, 1, 200)
y = np.linspace(0, 1, 200)
X_grid, Y_grid = np.meshgrid(x, y)
image = (
    np.sin(5 * np.pi * X_grid) * np.cos(3 * np.pi * Y_grid)
    + 0.5 * X_grid
    + 0.3 * Y_grid
)

# Full SVD
U, sigma, Vt = np.linalg.svd(image, full_matrices=False)

# Compress at different ranks
ranks = [1, 5, 10, 20, 50]

fig, axes = plt.subplots(1, len(ranks) + 1, figsize=(18, 3))

axes[0].imshow(image, cmap='gray')
axes[0].set_title(f'Original\n(rank {np.linalg.matrix_rank(image)})')
axes[0].axis('off')

for i, k in enumerate(ranks):
    # Rank-k approximation
    image_k = U[:, :k] @ np.diag(sigma[:k]) @ Vt[:k, :]
    axes[i + 1].imshow(image_k, cmap='gray')
    # Compression ratio: original has m*n values, compressed has k*(m+n+1)
    m, n = image.shape
    compression = (k * (m + n + 1)) / (m * n) * 100
    axes[i + 1].set_title(f'Rank {k}\n({compression:.1f}% storage)')
    axes[i + 1].axis('off')

plt.suptitle('Low-Rank Approximation with SVD', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Plot singular value decay -- shows how much information each component carries
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sigma, 'o-', markersize=3)
axes[0].set_xlabel('Component index')
axes[0].set_ylabel('Singular value')
axes[0].set_title('Singular Value Spectrum')
axes[0].grid(True, alpha=0.3)

cumulative = np.cumsum(sigma**2) / np.sum(sigma**2)
axes[1].plot(cumulative, 'o-', markersize=3)
axes[1].axhline(0.99, color='red', linestyle='--', label='99% variance')
axes[1].set_xlabel('Number of components')
axes[1].set_ylabel('Cumulative variance explained')
axes[1].set_title('Cumulative Variance Explained')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# How many components for 99% variance?
k_99 = np.searchsorted(cumulative, 0.99) + 1
print(f"Components needed for 99% variance: {k_99} out of {len(sigma)}")

### Exercise 5.1

Create your own $100 \times 100$ image matrix (try `np.random.randn(100, 100)` or a combination of `np.sin`/`np.cos` patterns). Compute its SVD and plot the rank-1, rank-5, and rank-20 approximations side by side. Which rank gives a visually acceptable result?

In [ ]:
# YOUR CODE HERE


### Exercise 5.2

For the image you created above, plot the cumulative variance explained curve. How many singular values do you need to capture 95% of the total variance?

In [ ]:
# YOUR CODE HERE


---
## Self-Assessment

After completing this notebook, you should be able to answer:

1. **What does it mean geometrically for a vector to be an eigenvector of a matrix?** (Hint: what happens to its direction?)

2. **Why do we prefer `np.linalg.eigh` over `np.linalg.eig` for covariance matrices?** (Hint: two reasons -- performance and guarantees.)

3. **If a data matrix $X$ has shape $(1000, 50)$, what are the shapes of $U$, $\Sigma$, and $V^T$ from its SVD?** How many singular values are there?

4. **In a rank-$k$ SVD approximation, what determines how large $k$ needs to be?** (Hint: think about how fast the singular values decay and how much variance you want to preserve.)

---
## References

1. **3Blue1Brown -- Essence of Linear Algebra** (video series)  
   [https://www.3blue1brown.com/topics/linear-algebra](https://www.3blue1brown.com/topics/linear-algebra)  
   Excellent visual intuition for eigenvalues, eigenvectors, and change of basis. Start with Chapter 14 (Eigenvectors and eigenvalues).

2. **NumPy Linear Algebra documentation**  
   [https://numpy.org/doc/stable/reference/routines.linalg.html](https://numpy.org/doc/stable/reference/routines.linalg.html)  
   Reference for `np.linalg.eig`, `np.linalg.eigh`, `np.linalg.svd`, and related functions.

3. **Gilbert Strang -- Linear Algebra and Its Applications**  
   Classic textbook. Chapters on eigenvalues (Ch. 6) and SVD (Ch. 7) are particularly relevant. MIT OCW lectures available at [https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/).